# Data

In [10]:
import json

with open('data/result.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

In [11]:
import pandas as pd

df = pd.DataFrame(data['messages'])

In [12]:
from utils import preprocess_df

df = preprocess_df(df)

Загружено сообщений для анализа: 32738


# Pipeline

In [13]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
import umap

reducer = umap.UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)

In [15]:
from sklearn.cluster import DBSCAN

clusterer = DBSCAN(eps=0.005, min_samples=3)

In [16]:
from bertopic import BERTopic

topic_model = BERTopic(
    embedding_model=model,
    hdbscan_model=clusterer,
    umap_model=reducer,
    verbose=True
)

topics, probs = topic_model.fit_transform(df['clean_text'].tolist())

2026-03-29 14:41:29,635 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/1024 [00:00<?, ?it/s]

2026-03-29 14:43:15,184 - BERTopic - Embedding - Completed ✓
2026-03-29 14:43:15,185 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-29 14:43:48,222 - BERTopic - Dimensionality - Completed ✓
2026-03-29 14:43:48,224 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-29 14:43:48,542 - BERTopic - Cluster - Completed ✓
2026-03-29 14:43:48,551 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-29 14:43:48,896 - BERTopic - Representation - Completed ✓


# Analysis

In [17]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,28405,-1_не_на_ты_мне,"[не, на, ты, мне, бля, что, ну, он, это, меня]","[ну что что, не бля у тебя там реально, что эт..."
1,0,57,0_курить_сигареты_курит_куришь,"[курить, сигареты, курит, куришь, крек, курени...","[Ты курил сигареты, курить хочу, Курить]"
2,1,51,1_gyrozeppeli2_попали_рест_открыла,"[gyrozeppeli2, попали, рест, открыла, тусит, п...","[@Gyrozeppeli2, @Gyrozeppeli2 😉😉😉, @Gyrozeppeli2]"
3,2,46,2_извини_извините_прости_прощаю,"[извини, извините, прости, прощаю, извинился, ...","[Извини, извини, Извини]"
4,3,44,3_esktpnk_киря_объяснись_сдавай,"[esktpnk, киря, объяснись, сдавай, подскажи, о...","[@esktpnk, @esktpnk, @esktpnk]"
...,...,...,...,...,...
598,597,3,597_дома_сих_пор_до,"[дома, сих, пор, до, бля, , , , , ]","[я дома, я дома, бля до сих пор дома]"
599,598,3,598_шнейне___,"[шнейне, , , , , , , , , ]","[Шнейне, Шнейне, Шнейне]"
600,599,3,599_могу_но_не_,"[могу, но, не, , , , , , , ]","[Но не могу, Не могу, Не могу]"
601,600,3,600_зацените___,"[зацените, , , , , , , , , ]","[Зацените, Зацените, Зацените]"


In [18]:
topic_model.get_topic_info().describe()

,Topic,Count
count,603.000000,603.000000
mean,300.000000,54.291874
std,174.215384,1156.468177
min,-1.000000,3.000000
25%,149.500000,3.000000
50%,300.000000,4.000000
75%,450.500000,8.000000
max,601.000000,28405.000000
